# Lab Task 5: Polynomial Regression
**Goal:** Apply Polynomial Regression on `Position_Salaries.csv` and display the
result, showing how it overcomes the limits of simple Linear Regression on
non-linear (curved) data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures


## Step 1: Upload and load the dataset

In [ ]:
from google.colab import files
uploaded = files.upload()   # choose Position_Salaries.csv


In [ ]:
dataset = pd.read_csv("Position_Salaries.csv")
dataset


In [ ]:
# We use 'Level' (1-10) as our feature (X) and 'Salary' as target (y).
# 'Position' is just a text label for the level, so we skip it as a feature.
X = dataset.iloc[:, 1:2].values   # Level column, kept 2-D using 1:2 slicing
y = dataset.iloc[:, 2].values     # Salary column


## Step 2: First try SIMPLE Linear Regression (to see why it fails)
Salary vs Level is clearly a CURVE, not a straight line (a Business Analyst to CEO
jump is not a constant salary increase per level) -- so a straight line will fit
this data poorly.

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X, y)

plt.scatter(X, y, color='red')
plt.plot(X, lin_reg.predict(X), color='blue')
plt.title('Truth or Bluff (Simple Linear Regression)')
plt.xlabel('Position Level')
plt.ylabel('Salary')
plt.show()


**Observation:** The straight blue line badly under/over-shoots the salary at
almost every level -- linear regression cannot capture this curved relationship.
This is exactly the problem Polynomial Regression solves.

## Step 3: Apply Polynomial Regression
We transform the single feature `Level` into polynomial terms
(Level, Level^2, Level^3, Level^4 ...) using `PolynomialFeatures`, then fit an
ordinary Linear Regression on TOP of those new polynomial features.
This is why it's still technically "Linear" Regression -- it's linear in the
coefficients, even though the resulting curve is not a straight line.

In [ ]:
poly_reg = PolynomialFeatures(degree=4)   # degree=4 gives a good curve fit for this dataset
X_poly = poly_reg.fit_transform(X)         # transforms X into [1, X, X^2, X^3, X^4]

lin_reg_2 = LinearRegression()
lin_reg_2.fit(X_poly, y)


## Step 4: Visualise the Polynomial Regression results

In [ ]:
# Use a finer-grained X range for a smoother curve
X_grid = np.arange(X.min(), X.max(), 0.1).reshape(-1, 1)

plt.scatter(X, y, color='red')
plt.plot(X_grid, lin_reg_2.predict(poly_reg.transform(X_grid)), color='blue')
plt.title('Truth or Bluff (Polynomial Regression)')
plt.xlabel('Position Level')
plt.ylabel('Salary')
plt.show()


**Observation:** The blue curve now follows the actual salary data points very
closely -- this is the "curvilinear" fit that polynomial regression provides.

## Step 5: Predict a new result
Example: predict the salary for Level 6.5 (halfway between Region Manager and
Partner) using BOTH models, to directly compare their predictions.

In [ ]:
linear_prediction = lin_reg.predict([[6.5]])
print("Simple Linear Regression prediction for Level 6.5:", linear_prediction[0])

poly_prediction = lin_reg_2.predict(poly_reg.transform([[6.5]]))
print("Polynomial Regression prediction for Level 6.5:", poly_prediction[0])


## Conclusion
Simple linear regression gives an unrealistic (usually too low or too high) salary
estimate for level 6.5 because it forces a straight line through curved data.
Polynomial regression, by adding higher-degree terms of `Level`, captures the true
curvilinear trend and gives a far more believable prediction.